In [4]:
import pandas as pd
from stable_baselines3 import PPO
import numpy as np


def _state_to_obs(state):
    obs = {
        'Time': np.array([state[-2]], dtype=np.float32),
        'User': int(state[-1]),
        # 'shift': int(state[-2][1])-1,
        # 'dayofweek': int(state[-1])

    }
    return obs

In [27]:
df_test = pd.read_csv("../data/cooked_test.csv")
model = PPO.load("../new_data1.zip")

mix = 0
add = 0
cont = 0
total = len(df_test)
mix1_history = []
mix2_history = []
mix3_history = []
mix4_history = []
add_history = []
cont_history = []

In [28]:
action_list = []
for data in df_test[['hora_decimal', 'encoded_user']].values: 
    # print(data)
    # print(_state_to_obs(data))
    action = list(model.predict(_state_to_obs(data)))[0]
    action_list.append(action)
    prediction = [action[0],
                  action[1],
                  action[2], 
                  action[3], 
                  action[-2], 
                  action[-1]]  # [mix, additive, container]

    mix1_history.append(prediction[0])
    mix2_history.append(prediction[1])
    mix3_history.append(prediction[2])
    mix4_history.append(prediction[3])
    
    add_history.append(prediction[-2])
    cont_history.append(prediction[-1])
    selection = list(df_test[(df_test['encoded_user'] == data[1]) & (df_test['hora_decimal'] == data[0])][['encoded_mixture','additive','encoded_container']].iloc[0])
    if selection[0] == prediction[0]:
        mix += 1
    if selection[1] == prediction[-2]:
        add += 1
    if selection[2] == prediction[-1]:
        cont += 1

In [29]:
dicc_history = {'Mix 1': mix1_history,
                'Mix 2': mix2_history,
                'Mix 3': mix3_history,
                'Mix 4': mix4_history, 
                'Additive': add_history, 
                'Container': cont_history}
df_history = pd.DataFrame(dicc_history)
df_history = pd.concat([df_test[['encoded_user','hora_decimal']], df_history], axis=1)
df_actions = pd.DataFrame(action_list)
print(df_history.describe())
print(f'Mix score:{mix/total}, Additive score: {add/total}, Container score: {cont/total}', )#/len(df_test))

       encoded_user  hora_decimal       Mix 1       Mix 2       Mix 3  Mix 4  \
count    352.000000    352.000000  352.000000  352.000000  352.000000  352.0   
mean      14.781250     10.221148    7.213068   15.963068   15.798295   16.0   
std        9.960915      2.642626    4.599957    0.692902    0.958808    0.0   
min        0.000000      4.653333    0.000000    3.000000    1.000000   16.0   
25%        7.000000      8.377153    2.000000   16.000000   16.000000   16.0   
50%       17.000000      9.929722   10.000000   16.000000   16.000000   16.0   
75%       25.000000     12.104097   10.000000   16.000000   16.000000   16.0   
max       30.000000     16.575278   12.000000   16.000000   16.000000   16.0   

         Additive   Container  
count  352.000000  352.000000  
mean     1.303977    0.392045  
std      1.236292    0.488902  
min      0.000000    0.000000  
25%      0.000000    0.000000  
50%      1.000000    0.000000  
75%      3.000000    1.000000  
max      3.000000    1.